# 04 · Auto-label with open-vocabulary detection

**Agenda: 55–75 min.** No one is going to hand-draw boxes on 5,000 robot frames. An open-vocabulary detector (Grounding DINO, prompted with plain words) does the first pass; humans review the queue.

Prompts used: `robot gripper . robot arm . blue brick . scissors . drawer . cloth . cardboard box`, folded onto 7 labels. The result is on every frame as `auto_labels`.

In [ ]:
import fiftyone as fo
from fiftyone import ViewField as F

frames = fo.load_dataset("droid-frames-workshop")
print(frames.count_values("auto_labels.detections.label"))
session = fo.launch_app(frames)

## Look before you trust

Sort by number of detections, look at the empty frames and the crowded ones. Confidence is stored per box — filter it in the sidebar.

In [ ]:
n_dets = F("auto_labels.detections").length()
print("frames with no labels:", len(frames.match(n_dets == 0)))
print("frames with 8+ labels:", len(frames.match(n_dets >= 8)))

# Gripper boxes only, low confidence first: the ones a reviewer should check
gripper_low = (frames.filter_labels("auto_labels", (F("label") == "gripper") & (F("confidence") < 0.4))
                     .sort_by(F("auto_labels.detections").length(), reverse=True))
session.view = gripper_low

## Sanity check the labels against robot state

The gripper is visible in almost every wrist-camera frame. If the labeler misses it there, that is a labeler problem, not a data problem.

In [ ]:
wrist = frames.match(F("camera") == "wrist")
has_gripper = wrist.filter_labels("auto_labels", F("label") == "gripper", only_matches=True)
print(f"wrist frames: {len(wrist)}, with a gripper box: {len(has_gripper)} ({100*len(has_gripper)/len(wrist):.0f}%)")

# Brick boxes should live on the brick-in-drawer episodes. Where does the labeler actually put them?
brick = frames.filter_labels("auto_labels", F("label") == "brick", only_matches=True)
print(brick.count_values("task_type"))
session.view = brick.match(F("task_type") == "clump-unclump")

Most "brick" boxes land on **clump-unclump** episodes — there is no brick there. The prompt said *blue brick*; the labeler grabbed blue cloth and plush toys. That is a prompt problem, and it is exactly the kind of thing that only shows up when labels are checked against metadata you already have (the task string). Fix the prompt, or drop `brick` from the training classes for those tasks.

## Optional: run Grounding DINO yourself on 20 frames (≈30 s on Apple Silicon, ~2 min CPU)

In [ ]:
# OPTIONAL
# import torch
# from PIL import Image
# from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
# mid = "IDEA-Research/grounding-dino-tiny"
# proc = AutoProcessor.from_pretrained(mid); model = AutoModelForZeroShotObjectDetection.from_pretrained(mid).eval()
# text = "robot gripper . blue brick . scissors ."
# for s in frames.take(20, seed=7).iter_samples(autosave=True):
#     im = Image.open(s.filepath).convert("RGB"); w, h = im.size
#     inp = proc(images=im, text=text, return_tensors="pt")
#     with torch.no_grad(): out = model(**inp)
#     r = proc.post_process_grounded_object_detection(out, inp.input_ids, threshold=0.3, text_threshold=0.25, target_sizes=[(h, w)])[0]
#     s["my_labels"] = fo.Detections(detections=[
#         fo.Detection(label=str(l), confidence=float(c), bounding_box=[b[0]/w, b[1]/h, (b[2]-b[0])/w, (b[3]-b[1])/h])
#         for l, c, b in zip(r["text_labels"] if "text_labels" in r else r["labels"], r["scores"], r["boxes"].tolist())])
# session.view = frames.exists("my_labels")

### The training set

`split` was assigned **by episode** (train 60 / val 15 episodes) so validation frames come from scenes the model never saw. Near-duplicates are excluded from training. This is what went to Nebius in notebook 05.

In [ ]:
print(frames.count_values("split"))
train = frames.match(F("split") == "train").match_tags("near-duplicate", bool=False)
print(len(train), "training frames")